In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_loan_data.csv', parse_dates=['origination_date'])
df.shape

(10000, 17)

In [2]:
df['default_flag'].value_counts(normalize=True)

default_flag
0    0.9363
1    0.0637
Name: proportion, dtype: float64

In [3]:
df['credit_score_band'] = pd.cut(
    df['orig_credit_score'],
    bins=[300, 580, 650, 700, 750, 850],
    labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
)

In [4]:
df['balance_to_original_ratio'] = (df['current_balance'] / df['original_balance']).round(3)

df['high_utilization_flag'] = np.where(df['utilization'] > 0.8, 1, 0)

df['score_deteriorated_flag'] = np.where(df['credit_score_change'] < -30, 1, 0)

In [5]:
df['dpd_bucket'] = pd.cut(
    df['days_past_due'],
    bins=[-1, 0, 30, 60, 90, 180],
    labels=['Current', '1-30', '31-60', '61-90', '90+']
)

df['ever_watchlisted'] = df['watchlist_flag']

In [6]:
categorical_cols = ['loan_type', 'region', 'loan_purpose', 'credit_score_band', 'dpd_bucket']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [7]:
feature_cols = [
    'orig_credit_score', 'current_credit_score', 'credit_score_change',
    'original_balance', 'current_balance', 'balance_to_original_ratio',
    'interest_rate', 'months_on_book', 'days_past_due',
    'watchlist_flag', 'high_utilization_flag', 'score_deteriorated_flag'
] + [c for c in df_encoded.columns if c.startswith(('loan_type_', 'region_', 'loan_purpose_', 'credit_score_band_', 'dpd_bucket_'))]

model_df = df_encoded[['loan_id', 'default_flag'] + feature_cols]
model_df.shape

(10000, 29)

Note: days_past_due and default_flag are mechanically linked in the data generation process (90+ DPD forces default_flag=1). This is a known simplification for this synthetic project. In a production PD model, DPD-based features would typically be excluded or lagged to avoid
target leakage, since the model should predict default before it happens, not from concurrent delinquency data.